# DBSQL JSON & VARIANT Demo — Setup & Data Generation

This notebook creates four schemas and loads ~1,400 rows of synthetic healthcare data
(claims, member profiles, provider network, operational events).  Run it **once** before
opening `01_json_demo.ipynb`.

**Estimated run time: 3–5 minutes on a Small serverless SQL warehouse.**

---

## ⚠️  Before you run anything — set your catalog name

The cell below creates a widget at the top of this notebook.
**Enter your Unity Catalog catalog name in the widget field before running any SQL cells.**

If you are unsure of your catalog name, run:
```sql
SHOW CATALOGS;
```
in a separate query window and pick the one you have write access to.


In [ ]:
dbutils.widgets.text("catalog_name", "", "⚠️ Enter your catalog name here (e.g. main)")
catalog_name = dbutils.widgets.get("catalog_name")

if not catalog_name.strip():
    raise ValueError(
        "\n\n❌  catalog_name widget is empty.\n"
        "    Enter your catalog name in the widget at the top of the notebook,\n"
        "    then re-run this cell before continuing.\n"
    )
print(f"✅  Catalog set to: {catalog_name}")


In [ ]:
%sql
USE CATALOG ${catalog_name};

## Step 1 — Create schemas

Four schemas, one per domain.  They will all live inside the catalog you specified above.


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.claims_json_demo
  COMMENT 'Claims submission and adjudication data with JSON payloads for DBSQL JSON demo';

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.member_json_demo
  COMMENT 'Member profile and engagement data with JSON payloads for DBSQL JSON demo';

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.provider_json_demo
  COMMENT 'Provider directory and network data with VARIANT payloads for DBSQL JSON demo';

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.ops_json_demo
  COMMENT 'Operational events and workflow data with JSON payloads for DBSQL JSON demo';


## Step 2 — Claims submissions (500 rows)

Each row has structured columns plus a `claim_json` STRING column containing the full
claim payload: subscriber, billing provider, service lines array (1–4 lines each),
and adjudication detail.  This is the primary teaching table for JSON string extraction.


In [ ]:
%sql
DROP TABLE IF EXISTS ${catalog_name}.claims_json_demo.claims_submissions;

CREATE TABLE ${catalog_name}.claims_json_demo.claims_submissions (
  claim_id            STRING       COMMENT 'Unique claim identifier (CLM-YYYY-NNNNN)',
  submitted_date      DATE         COMMENT 'Date the claim was submitted to the payer',
  payer_id            STRING       COMMENT 'Payer/plan identifier',
  claim_type          STRING       COMMENT 'professional or institutional',
  total_charge_amount DECIMAL(12,2) COMMENT 'Header-level total charge',
  claim_json          STRING       COMMENT 'Full claim payload as JSON string: subscriber, billing_provider, service_lines array, adjudication'
) COMMENT 'Claims submissions with JSON payloads containing service lines, adjudication, and provider details';


In [ ]:
%sql
INSERT INTO ${catalog_name}.claims_json_demo.claims_submissions
WITH claim_ids AS (
  SELECT explode(sequence(1, 500)) AS rn
),
base AS (
  SELECT
    rn,
    concat('CLM-2024-', lpad(cast(rn AS STRING), 5, '0')) AS claim_id,
    date_add('2024-01-01', abs(hash(rn, 'dt')) % 365) AS submitted_date,
    concat('HUM-', lpad(cast(abs(hash(rn, 'pay')) % 20 + 1 AS STRING), 3, '0')) AS payer_id,
    CASE WHEN abs(hash(rn, 'ct')) % 10 < 6 THEN 'professional' ELSE 'institutional' END AS claim_type,
    concat('MBR-', lpad(cast(abs(hash(rn, 'mbr')) % 300 + 1 AS STRING), 6, '0')) AS member_id,
    concat('GRP-', lpad(cast(abs(hash(rn, 'grp')) % 50 + 1 AS STRING), 4, '0')) AS group_number,
    concat('1', lpad(cast(abs(hash(rn, 'bnpi')) % 999999999 + 1000000000 AS STRING), 9, '0')) AS billing_npi,
    CASE abs(hash(rn, 'fc')) % 5
      WHEN 0 THEN '09' WHEN 1 THEN '11' WHEN 2 THEN '12' WHEN 3 THEN '17' ELSE '22'
    END AS filing_code,
    abs(hash(rn, 'cob')) % 100 < 15 AS has_cob,
    CASE
      WHEN abs(hash(rn, 'adj')) % 100 < 75 THEN 'paid'
      WHEN abs(hash(rn, 'adj')) % 100 < 90 THEN 'denied'
      ELSE 'pending'
    END AS adj_status,
    abs(hash(rn, 'nl')) % 4 + 1 AS num_lines,
    rn AS seed
  FROM claim_ids
),
with_lines AS (
  SELECT b.*, le.line_num
  FROM base b
  LATERAL VIEW explode(sequence(1, b.num_lines)) le AS line_num
),
service_lines AS (
  SELECT
    seed,
    collect_list(
      concat(
        '{"line_number":', cast(line_num AS STRING),
        ',"procedure_code":"',
        CASE abs(hash(seed, 'cpt', line_num)) % 12
          WHEN 0 THEN '99213' WHEN 1 THEN '99214' WHEN 2 THEN '99215'
          WHEN 3 THEN '99283' WHEN 4 THEN '99284' WHEN 5 THEN '99285'
          WHEN 6 THEN '27447' WHEN 7 THEN '43239' WHEN 8 THEN '93000'
          WHEN 9 THEN '71046' WHEN 10 THEN '80053' ELSE '36415'
        END,
        '","modifiers":',
        CASE
          WHEN abs(hash(seed, 'mod', line_num)) % 10 < 4
          THEN concat('["', CASE abs(hash(seed, 'md1', line_num)) % 4 WHEN 0 THEN '25' WHEN 1 THEN '59' WHEN 2 THEN 'TC' ELSE '26' END, '"]')
          WHEN abs(hash(seed, 'mod', line_num)) % 10 < 6
          THEN concat('["',
            CASE abs(hash(seed, 'md2', line_num)) % 3 WHEN 0 THEN '25' WHEN 1 THEN '59' ELSE 'LT' END,
            '","',
            CASE abs(hash(seed, 'md3', line_num)) % 3 WHEN 0 THEN '76' WHEN 1 THEN 'XE' ELSE 'XS' END,
            '"]')
          ELSE '[]'
        END,
        ',"diagnosis_pointers":["',
        CASE abs(hash(seed, 'dx1', line_num)) % 6
          WHEN 0 THEN 'E11.9' WHEN 1 THEN 'I10' WHEN 2 THEN 'M17.11'
          WHEN 3 THEN 'J06.9' WHEN 4 THEN 'K21.0' ELSE 'Z00.00'
        END,
        CASE WHEN abs(hash(seed, 'dx2', line_num)) % 2 = 0
          THEN concat('","', CASE abs(hash(seed, 'dx2v', line_num)) % 5
              WHEN 0 THEN 'E78.5' WHEN 1 THEN 'I25.10' WHEN 2 THEN 'N18.3'
              WHEN 3 THEN 'G89.29' ELSE 'Z87.891' END)
          ELSE ''
        END,
        '"],"units":', cast(abs(hash(seed, 'u', line_num)) % 3 + 1 AS STRING),
        ',"charge_amount":', cast(round((abs(hash(seed, 'chg', line_num)) % 250000) / 100.0 + 50, 2) AS STRING),
        ',"place_of_service":"',
        CASE abs(hash(seed, 'pos', line_num)) % 5
          WHEN 0 THEN '11' WHEN 1 THEN '21' WHEN 2 THEN '22' WHEN 3 THEN '23' ELSE '81'
        END,
        '","date_of_service":"', cast(date_add(submitted_date, -(abs(hash(seed, 'dos', line_num)) % 5)) AS STRING),
        '","rendering_provider_npi":"', concat('1', lpad(cast(abs(hash(seed, 'rnpi', line_num)) % 999999999 + 1000000000 AS STRING), 9, '0')), '"',
        CASE WHEN claim_type = 'institutional'
          THEN concat(',"revenue_code":"',
            CASE abs(hash(seed, 'rev', line_num)) % 5
              WHEN 0 THEN '0120' WHEN 1 THEN '0250' WHEN 2 THEN '0320' WHEN 3 THEN '0450' ELSE '0510'
            END, '"')
          ELSE ''
        END,
        '}'
      )
    ) AS svc_lines_arr
  FROM with_lines
  GROUP BY seed, submitted_date, claim_type
)
SELECT
  b.claim_id,
  b.submitted_date,
  b.payer_id,
  b.claim_type,
  CAST(0 AS DECIMAL(12,2)) AS total_charge_amount,
  concat(
    '{"claim_number":"', b.claim_id,
    '","patient_control_number":"PCN-', lpad(cast(b.seed AS STRING), 8, '0'),
    '","filing_code":"', b.filing_code,
    '","release_of_info":"Y","coordination_of_benefits":', CASE WHEN b.has_cob THEN 'true' ELSE 'false' END,
    ',"subscriber":{"member_id":"', b.member_id,
    '","group_number":"', b.group_number,
    '","relationship_code":"', CASE abs(hash(b.seed, 'rel')) % 3 WHEN 0 THEN '18' WHEN 1 THEN '01' ELSE '19' END,
    '"},"billing_provider":{"npi":"', b.billing_npi,
    '","tax_id":"', concat(lpad(cast(abs(hash(b.seed, 'tax1')) % 90 + 10 AS STRING), 2, '0'), '-', lpad(cast(abs(hash(b.seed, 'tax2')) % 9000000 + 1000000 AS STRING), 7, '0')),
    '","name":"',
    CASE abs(hash(b.seed, 'bname')) % 8
      WHEN 0 THEN 'Sunrise Medical Group'
      WHEN 1 THEN 'Bluegrass Family Practice'
      WHEN 2 THEN 'Commonwealth Orthopedics'
      WHEN 3 THEN 'River City Cardiology'
      WHEN 4 THEN 'Heritage Health Partners'
      WHEN 5 THEN 'Appalachian Wellness Center'
      WHEN 6 THEN 'Derby City Internal Medicine'
      ELSE 'Southern Specialty Associates'
    END,
    '"},"service_lines":[', concat_ws(',', sl.svc_lines_arr),
    '],"adjudication":{"status":"', b.adj_status,
    '","paid_amount":',
    CASE WHEN b.adj_status = 'denied' THEN '0.00'
         WHEN b.adj_status = 'pending' THEN 'null'
         ELSE cast(round((abs(hash(b.seed, 'pa')) % 200000) / 100.0 + 100, 2) AS STRING)
    END,
    ',"allowed_amount":',
    CASE WHEN b.adj_status = 'pending' THEN 'null'
         ELSE cast(round((abs(hash(b.seed, 'aa')) % 250000) / 100.0 + 100, 2) AS STRING)
    END,
    ',"denial_codes":',
    CASE WHEN b.adj_status = 'denied'
      THEN concat('["',
        CASE abs(hash(b.seed, 'dc1')) % 5
          WHEN 0 THEN 'CO-4' WHEN 1 THEN 'CO-16' WHEN 2 THEN 'CO-29' WHEN 3 THEN 'PR-1' ELSE 'CO-197'
        END,
        CASE WHEN abs(hash(b.seed, 'dc2')) % 100 < 30
          THEN concat('","', CASE abs(hash(b.seed, 'dc3')) % 3 WHEN 0 THEN 'CO-50' WHEN 1 THEN 'OA-23' ELSE 'PR-2' END, '"]')
          ELSE '"]' END)
      ELSE '[]'
    END,
    ',"remark_codes":',
    CASE WHEN b.adj_status != 'pending' AND abs(hash(b.seed, 'rc')) % 100 < 30
      THEN concat('["', CASE abs(hash(b.seed, 'rcv')) % 4 WHEN 0 THEN 'N362' WHEN 1 THEN 'N657' WHEN 2 THEN 'M15' ELSE 'N519' END, '"]')
      ELSE '[]'
    END,
    '}}'
  ) AS claim_json
FROM base b
JOIN service_lines sl ON sl.seed = b.seed;


In [ ]:
%sql
-- Fix total_charge_amount by summing service lines from the generated JSON
UPDATE ${catalog_name}.claims_json_demo.claims_submissions
SET total_charge_amount = COALESCE(
  CAST(
    AGGREGATE(
      from_json(claim_json:service_lines, 'ARRAY<STRUCT<charge_amount: DOUBLE>>'),
      DOUBLE(0),
      (acc, x) -> acc + coalesce(x.charge_amount, 0)
    ) AS DECIMAL(12,2)
  ), 0);


## Step 3 — Member profiles (300 rows)

Each member has a `profile_json` STRING column with deeply nested data: demographics,
risk scores, SDoH flags, an active conditions array, programs enrolled array, and an
engagement history array.


In [ ]:
%sql
DROP TABLE IF EXISTS ${catalog_name}.member_json_demo.member_profiles;

CREATE TABLE ${catalog_name}.member_json_demo.member_profiles (
  member_id             STRING  COMMENT 'Unique member identifier (MBR-NNNNNN)',
  enrollment_start_date DATE    COMMENT 'Coverage effective date',
  plan_code             STRING  COMMENT 'Plan/product code',
  line_of_business      STRING  COMMENT 'Medicare Advantage, Commercial, or Medicaid',
  profile_json          STRING  COMMENT 'Rich member profile JSON: demographics, risk_scores, sdoh_flags, conditions array, programs_enrolled array, engagement_history array, preferred_communication'
) COMMENT 'Member profiles with deeply nested JSON including demographics, risk, SDoH, conditions, and engagement history';


In [ ]:
%sql
INSERT INTO ${catalog_name}.member_json_demo.member_profiles
WITH member_ids AS (SELECT explode(sequence(1, 300)) AS rn),
base AS (
  SELECT rn,
    concat('MBR-', lpad(cast(rn AS STRING), 6, '0')) AS member_id,
    date_add('2020-01-01', abs(hash(rn, 'enrl')) % 1460) AS enrollment_start_date,
    CASE abs(hash(rn, 'lob')) % 3
      WHEN 0 THEN 'Medicare Advantage' WHEN 1 THEN 'Commercial' ELSE 'Medicaid'
    END AS line_of_business,
    rn AS seed
  FROM member_ids
),
names_data AS (
  SELECT b.*,
    CASE abs(hash(seed, 'fn')) % 20
      WHEN 0 THEN 'James' WHEN 1 THEN 'Mary' WHEN 2 THEN 'Robert' WHEN 3 THEN 'Patricia'
      WHEN 4 THEN 'John' WHEN 5 THEN 'Jennifer' WHEN 6 THEN 'Michael' WHEN 7 THEN 'Linda'
      WHEN 8 THEN 'David' WHEN 9 THEN 'Elizabeth' WHEN 10 THEN 'William' WHEN 11 THEN 'Barbara'
      WHEN 12 THEN 'Richard' WHEN 13 THEN 'Susan' WHEN 14 THEN 'Joseph' WHEN 15 THEN 'Jessica'
      WHEN 16 THEN 'Thomas' WHEN 17 THEN 'Sarah' WHEN 18 THEN 'Charles' ELSE 'Karen'
    END AS fname,
    CASE abs(hash(seed, 'ln')) % 20
      WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams' WHEN 3 THEN 'Brown'
      WHEN 4 THEN 'Jones' WHEN 5 THEN 'Garcia' WHEN 6 THEN 'Miller' WHEN 7 THEN 'Davis'
      WHEN 8 THEN 'Rodriguez' WHEN 9 THEN 'Martinez' WHEN 10 THEN 'Hernandez' WHEN 11 THEN 'Lopez'
      WHEN 12 THEN 'Wilson' WHEN 13 THEN 'Anderson' WHEN 14 THEN 'Taylor' WHEN 15 THEN 'Thomas'
      WHEN 16 THEN 'Moore' WHEN 17 THEN 'Jackson' WHEN 18 THEN 'Martin' ELSE 'Lee'
    END AS lname,
    CASE abs(hash(seed, 'city')) % 15
      WHEN 0 THEN 'Louisville' WHEN 1 THEN 'Lexington' WHEN 2 THEN 'Bowling Green'
      WHEN 3 THEN 'Covington' WHEN 4 THEN 'Frankfort' WHEN 5 THEN 'Richmond'
      WHEN 6 THEN 'Georgetown' WHEN 7 THEN 'Florence' WHEN 8 THEN 'Elizabethtown'
      WHEN 9 THEN 'Owensboro' WHEN 10 THEN 'Paducah' WHEN 11 THEN 'Ashland'
      WHEN 12 THEN 'Henderson' WHEN 13 THEN 'Radcliff' ELSE 'Hopkinsville'
    END AS city,
    CASE abs(hash(seed, 'city')) % 15
      WHEN 0 THEN '40202' WHEN 1 THEN '40507' WHEN 2 THEN '42101' WHEN 3 THEN '41011'
      WHEN 4 THEN '40601' WHEN 5 THEN '40475' WHEN 6 THEN '40324' WHEN 7 THEN '41042'
      WHEN 8 THEN '42701' WHEN 9 THEN '42301' WHEN 10 THEN '42001' WHEN 11 THEN '41101'
      WHEN 12 THEN '42420' WHEN 13 THEN '40160' ELSE '42240'
    END AS zip
  FROM base b
)
SELECT member_id, enrollment_start_date,
  concat(CASE line_of_business WHEN 'Medicare Advantage' THEN 'MA-' WHEN 'Commercial' THEN 'COM-' ELSE 'MCD-' END,
    lpad(cast(abs(hash(seed, 'pc')) % 20 + 1 AS STRING), 3, '0')) AS plan_code,
  line_of_business,
  concat(
    '{"demographics":{"first_name":"', fname, '","last_name":"', lname,
    '","date_of_birth":"', cast(date_add('1940-01-01', abs(hash(seed, 'dob')) % 25000) AS STRING),
    '","gender":"', CASE WHEN abs(hash(seed, 'gen')) % 100 < 48 THEN 'M' ELSE 'F' END,
    '","language":"', CASE abs(hash(seed, 'lang')) % 5 WHEN 0 THEN 'English' WHEN 1 THEN 'Spanish' WHEN 2 THEN 'English' WHEN 3 THEN 'English' ELSE 'Vietnamese' END,
    '","address":{"street":"', cast(abs(hash(seed, 'stnum')) % 9899 + 100 AS STRING), ' ',
    CASE abs(hash(seed, 'stname')) % 6 WHEN 0 THEN 'Main St' WHEN 1 THEN 'Oak Ave' WHEN 2 THEN 'Elm Dr' WHEN 3 THEN 'Maple Ln' WHEN 4 THEN 'River Rd' ELSE 'Highland Blvd' END,
    '","city":"', city, '","state":"KY","zip":"', zip, '"}}',
    ',"risk_scores":{"hcc_score":', cast(round((abs(hash(seed, 'hcc')) % 3500) / 1000.0 + 0.5, 3) AS STRING),
    ',"rx_score":', cast(round((abs(hash(seed, 'rx')) % 2000) / 1000.0 + 0.2, 3) AS STRING),
    ',"sdoh_risk_level":"', CASE abs(hash(seed, 'sdoh')) % 4 WHEN 0 THEN 'low' WHEN 1 THEN 'moderate' WHEN 2 THEN 'high' ELSE 'low' END, '"}',
    ',"sdoh_flags":{"food_insecurity":', CASE WHEN abs(hash(seed, 'fi')) % 100 < 12 THEN 'true' ELSE 'false' END,
    ',"transportation_barrier":', CASE WHEN abs(hash(seed, 'tb')) % 100 < 18 THEN 'true' ELSE 'false' END,
    ',"housing_instability":', CASE WHEN abs(hash(seed, 'hi')) % 100 < 8 THEN 'true' ELSE 'false' END,
    ',"social_isolation":', CASE WHEN abs(hash(seed, 'si')) % 100 < 15 THEN 'true' ELSE 'false' END, '}',
    ',"conditions":[',
    CASE
      WHEN abs(hash(seed, 'cond')) % 100 < 30 THEN
        concat('{"code":"E11.9","description":"Type 2 diabetes mellitus","onset_date":"', cast(date_add('2018-01-01', abs(hash(seed, 'cdt1')) % 1500) AS STRING), '","status":"active"}',
          CASE WHEN abs(hash(seed, 'cond2')) % 2 = 0 THEN concat(',{"code":"I10","description":"Essential hypertension","onset_date":"', cast(date_add('2016-01-01', abs(hash(seed, 'cdt2')) % 2000) AS STRING), '","status":"active"}') ELSE '' END,
          CASE WHEN abs(hash(seed, 'cond3')) % 100 < 30 THEN concat(',{"code":"E78.5","description":"Hyperlipidemia","onset_date":"', cast(date_add('2019-01-01', abs(hash(seed, 'cdt3')) % 1200) AS STRING), '","status":"active"}') ELSE '' END)
      WHEN abs(hash(seed, 'cond')) % 100 < 50 THEN
        concat('{"code":"I10","description":"Essential hypertension","onset_date":"', cast(date_add('2017-01-01', abs(hash(seed, 'cdt4')) % 1800) AS STRING), '","status":"active"}',
          CASE WHEN abs(hash(seed, 'cond4')) % 100 < 40 THEN concat(',{"code":"I25.10","description":"Coronary artery disease","onset_date":"', cast(date_add('2019-06-01', abs(hash(seed, 'cdt5')) % 1000) AS STRING), '","status":"active"}') ELSE '' END)
      WHEN abs(hash(seed, 'cond')) % 100 < 70 THEN
        concat('{"code":"J44.1","description":"COPD with acute exacerbation","onset_date":"', cast(date_add('2020-01-01', abs(hash(seed, 'cdt6')) % 800) AS STRING), '","status":"active"}')
      ELSE ''
    END, ']',
    ',"programs_enrolled":[',
    concat_ws(',',
      CASE WHEN abs(hash(seed, 'p1')) % 100 < 25 THEN '"CHF_Management"' END,
      CASE WHEN abs(hash(seed, 'p2')) % 100 < 30 THEN '"DM2_Care"' END,
      CASE WHEN abs(hash(seed, 'p3')) % 100 < 20 THEN '"Annual_Wellness"' END,
      CASE WHEN abs(hash(seed, 'p4')) % 100 < 15 THEN '"COPD_Support"' END,
      CASE WHEN abs(hash(seed, 'p5')) % 100 < 10 THEN '"Behavioral_Health"' END,
      CASE WHEN abs(hash(seed, 'p6')) % 100 < 12 THEN '"Fall_Prevention"' END
    ), ']',
    ',"engagement_history":[',
    CASE WHEN abs(hash(seed, 'eng')) % 100 < 80 THEN
      concat('{"event_type":"outreach_call","channel":"phone","timestamp":"',
        cast(date_add('2024-01-01', abs(hash(seed, 'et1')) % 300) AS STRING), 'T',
        lpad(cast(abs(hash(seed, 'eh1')) % 12 + 8 AS STRING), 2, '0'), ':',
        lpad(cast(abs(hash(seed, 'em1')) % 60 AS STRING), 2, '0'), ':00Z',
        '","outcome":"', CASE abs(hash(seed, 'eo1')) % 4 WHEN 0 THEN 'completed' WHEN 1 THEN 'voicemail' WHEN 2 THEN 'no_answer' ELSE 'completed' END,
        '","agent_id":"AGT-', lpad(cast(abs(hash(seed, 'ea1')) % 50 + 1 AS STRING), 3, '0'), '"}',
        CASE WHEN abs(hash(seed, 'eng2')) % 100 < 60 THEN
          concat(',{"event_type":"', CASE WHEN abs(hash(seed, 'ety2')) % 2 = 0 THEN 'portal_message' ELSE 'sms_reminder' END,
            '","channel":"', CASE WHEN abs(hash(seed, 'ety2')) % 2 = 0 THEN 'portal' ELSE 'sms' END,
            '","timestamp":"', cast(date_add('2024-03-01', abs(hash(seed, 'et2')) % 200) AS STRING),
            'T14:30:00Z","outcome":"delivered","agent_id":null}')
        ELSE '' END,
        CASE WHEN abs(hash(seed, 'eng3')) % 100 < 30 THEN
          concat(',{"event_type":"care_mgmt_touch","channel":"phone","timestamp":"',
            cast(date_add('2024-06-01', abs(hash(seed, 'et3')) % 150) AS STRING),
            'T10:00:00Z","outcome":"completed","agent_id":"CM-',
            lpad(cast(abs(hash(seed, 'ea3')) % 20 + 1 AS STRING), 3, '0'), '"}')
        ELSE '' END)
    ELSE '' END, ']',
    ',"preferred_communication":{"channel":"',
    CASE abs(hash(seed, 'pcc')) % 4 WHEN 0 THEN 'phone' WHEN 1 THEN 'email' WHEN 2 THEN 'portal' ELSE 'sms' END,
    '","time_of_day":"', CASE abs(hash(seed, 'tod')) % 3 WHEN 0 THEN 'morning' WHEN 1 THEN 'afternoon' ELSE 'evening' END,
    '","opt_out_sms":', CASE WHEN abs(hash(seed, 'oos')) % 100 < 10 THEN 'true' ELSE 'false' END, '}}'
  ) AS profile_json
FROM names_data;


## Step 4 — Provider network (200 rows, VARIANT column)

This table uses the **VARIANT** type instead of STRING for its JSON column — demonstrating
`PARSE_JSON`, `variant_get`, and `variant_explode`.  Provider data includes name,
specialties, addresses, languages, network participation per plan, and contract details.


In [ ]:
%sql
DROP TABLE IF EXISTS ${catalog_name}.provider_json_demo.provider_network;

CREATE TABLE ${catalog_name}.provider_json_demo.provider_network (
  provider_id    STRING   COMMENT 'Internal provider identifier (PRV-NNNNNN)',
  npi            STRING   COMMENT 'National Provider Identifier (10-digit)',
  provider_type  STRING   COMMENT 'individual or organization',
  effective_date DATE     COMMENT 'Network effective date',
  provider_data  VARIANT  COMMENT 'Provider directory payload as VARIANT: name, specialties array, addresses array, languages array, network_participation array, contract object'
) COMMENT 'Provider directory with VARIANT column — showcases native semi-structured type';


In [ ]:
%sql
INSERT INTO ${catalog_name}.provider_json_demo.provider_network
WITH provider_ids AS (SELECT explode(sequence(1, 200)) AS rn),
base AS (
  SELECT rn,
    concat('PRV-', lpad(cast(rn AS STRING), 6, '0')) AS provider_id,
    concat('1', lpad(cast(abs(hash(rn, 'npi')) % 999999999 + 1000000000 AS STRING), 9, '0')) AS npi,
    CASE WHEN abs(hash(rn, 'pt')) % 10 < 7 THEN 'individual' ELSE 'organization' END AS provider_type,
    date_add('2020-01-01', abs(hash(rn, 'edt')) % 1460) AS effective_date,
    rn AS seed
  FROM provider_ids
)
SELECT provider_id, npi, provider_type, effective_date,
  PARSE_JSON(concat(
    CASE WHEN provider_type = 'individual' THEN
      concat('{"name":{"first":"',
        CASE abs(hash(seed, 'dfn')) % 15
          WHEN 0 THEN 'Sarah' WHEN 1 THEN 'James' WHEN 2 THEN 'Maria' WHEN 3 THEN 'David'
          WHEN 4 THEN 'Emily' WHEN 5 THEN 'Robert' WHEN 6 THEN 'Lisa' WHEN 7 THEN 'Michael'
          WHEN 8 THEN 'Anna' WHEN 9 THEN 'William' WHEN 10 THEN 'Rachel' WHEN 11 THEN 'Thomas'
          WHEN 12 THEN 'Karen' WHEN 13 THEN 'Daniel' ELSE 'Jennifer'
        END,
        '","last":"',
        CASE abs(hash(seed, 'dln')) % 15
          WHEN 0 THEN 'Patel' WHEN 1 THEN 'Chen' WHEN 2 THEN 'Kim' WHEN 3 THEN 'Nguyen'
          WHEN 4 THEN 'Shah' WHEN 5 THEN 'Kumar' WHEN 6 THEN 'Park' WHEN 7 THEN 'Ahmed'
          WHEN 8 THEN 'Gupta' WHEN 9 THEN 'Lee' WHEN 10 THEN 'Wang' WHEN 11 THEN 'Singh'
          WHEN 12 THEN 'Thompson' WHEN 13 THEN 'White' ELSE 'Adams'
        END,
        '","credentials":"', CASE abs(hash(seed, 'cred')) % 4 WHEN 0 THEN 'MD' WHEN 1 THEN 'DO' WHEN 2 THEN 'MD, FACP' ELSE 'DO, MBA' END, '"}')
    ELSE
      concat('{"name":{"organization_name":"',
        CASE abs(hash(seed, 'org')) % 8
          WHEN 0 THEN 'Humana Health Centers' WHEN 1 THEN 'Kentucky Medical Associates'
          WHEN 2 THEN 'Bluegrass Community Health' WHEN 3 THEN 'Commonwealth Care Network'
          WHEN 4 THEN 'River Valley Medical Group' WHEN 5 THEN 'Appalachian Regional Healthcare'
          WHEN 6 THEN 'Derby City Medical Center' ELSE 'Southern Health Partners'
        END, '"}')
    END,
    ',"specialties":[{"code":"',
    CASE abs(hash(seed, 'sp1')) % 10
      WHEN 0 THEN '207R00000X","description":"Internal Medicine'
      WHEN 1 THEN '207RC0000X","description":"Cardiovascular Disease'
      WHEN 2 THEN '207RE0101X","description":"Endocrinology'
      WHEN 3 THEN '207RG0100X","description":"Gastroenterology'
      WHEN 4 THEN '207RP1001X","description":"Pulmonary Disease'
      WHEN 5 THEN '208D00000X","description":"General Practice'
      WHEN 6 THEN '207Q00000X","description":"Family Medicine'
      WHEN 7 THEN '207X00000X","description":"Orthopedic Surgery'
      WHEN 8 THEN '2084N0400X","description":"Neurology'
      ELSE '207Y00000X","description":"Ophthalmology'
    END,
    '","board_certified":', CASE WHEN abs(hash(seed, 'bc1')) % 100 < 70 THEN 'true' ELSE 'false' END, '}',
    CASE WHEN abs(hash(seed, 'sp2f')) % 100 < 35 THEN
      concat(',{"code":"',
        CASE abs(hash(seed, 'sp2')) % 5
          WHEN 0 THEN '261QM0801X","description":"Mental Health Facility'
          WHEN 1 THEN '261QR0400X","description":"Rehabilitation Facility'
          WHEN 2 THEN '282N00000X","description":"General Acute Care Hospital'
          WHEN 3 THEN '261QU0200X","description":"Urgent Care Facility'
          ELSE '261QP2300X","description":"Primary Care Clinic'
        END,
        '","board_certified":', CASE WHEN abs(hash(seed, 'bc2')) % 100 < 60 THEN 'true' ELSE 'false' END, '}')
    ELSE '' END, ']',
    ',"addresses":[{"type":"practice","street":"', cast(abs(hash(seed, 'stnum')) % 9899 + 100 AS STRING), ' ',
    CASE abs(hash(seed, 'stname')) % 6
      WHEN 0 THEN 'Medical Center Dr' WHEN 1 THEN 'Healthcare Blvd' WHEN 2 THEN 'Clinic Way'
      WHEN 3 THEN 'Wellness Pkwy' WHEN 4 THEN 'Hospital Dr' ELSE 'Medical Park Ln'
    END,
    '","city":"',
    CASE abs(hash(seed, 'pcity')) % 10
      WHEN 0 THEN 'Louisville' WHEN 1 THEN 'Lexington' WHEN 2 THEN 'Bowling Green'
      WHEN 3 THEN 'Covington' WHEN 4 THEN 'Frankfort' WHEN 5 THEN 'Richmond'
      WHEN 6 THEN 'Georgetown' WHEN 7 THEN 'Florence' WHEN 8 THEN 'Elizabethtown'
      ELSE 'Owensboro'
    END,
    '","state":"KY","zip":"',
    CASE abs(hash(seed, 'pcity')) % 10
      WHEN 0 THEN '40202' WHEN 1 THEN '40507' WHEN 2 THEN '42101' WHEN 3 THEN '41011'
      WHEN 4 THEN '40601' WHEN 5 THEN '40475' WHEN 6 THEN '40324' WHEN 7 THEN '41042'
      WHEN 8 THEN '42701' ELSE '42301'
    END,
    '","phone":"502-555-', lpad(cast(abs(hash(seed, 'ph1')) % 9000 + 1000 AS STRING), 4, '0'), '"}',
    CASE WHEN abs(hash(seed, 'ma')) % 2 = 0 THEN
      concat(',{"type":"mailing","street":"PO Box ', cast(abs(hash(seed, 'pob')) % 9000 + 1000 AS STRING),
        '","city":"Louisville","state":"KY","zip":"40201","phone":"502-555-',
        lpad(cast(abs(hash(seed, 'ph2')) % 9000 + 1000 AS STRING), 4, '0'), '"}')
    ELSE '' END, ']',
    ',"languages":["English"',
    CASE WHEN abs(hash(seed, 'l1')) % 100 < 25 THEN ',"Spanish"' ELSE '' END,
    CASE WHEN abs(hash(seed, 'l2')) % 100 < 10 THEN ',"Vietnamese"' ELSE '' END,
    CASE WHEN abs(hash(seed, 'l3')) % 100 < 8 THEN ',"Arabic"' ELSE '' END, ']',
    ',"network_participation":[',
    concat_ws(',',
      concat('{"plan_code":"MA-001","network_tier":"',
        CASE abs(hash(seed, 'nt1')) % 3 WHEN 0 THEN 'tier1' WHEN 1 THEN 'tier2' ELSE 'tier1' END,
        '","accepting_new_patients":', CASE WHEN abs(hash(seed, 'anp1')) % 100 < 85 THEN 'true' ELSE 'false' END,
        ',"effective_date":"', cast(effective_date AS STRING), '"}'),
      CASE WHEN abs(hash(seed, 'np2')) % 100 < 60 THEN
        concat('{"plan_code":"COM-005","network_tier":"',
          CASE abs(hash(seed, 'nt2')) % 3 WHEN 0 THEN 'tier1' WHEN 1 THEN 'tier2' ELSE 'out_of_network' END,
          '","accepting_new_patients":', CASE WHEN abs(hash(seed, 'anp2')) % 100 < 80 THEN 'true' ELSE 'false' END,
          ',"effective_date":"', cast(effective_date AS STRING), '"}') END,
      CASE WHEN abs(hash(seed, 'np3')) % 100 < 40 THEN
        concat('{"plan_code":"MCD-010","network_tier":"tier1","accepting_new_patients":',
          CASE WHEN abs(hash(seed, 'anp3')) % 100 < 90 THEN 'true' ELSE 'false' END,
          ',"effective_date":"', cast(effective_date AS STRING), '"}') END
    ), ']',
    ',"contract":{"fee_schedule_id":"FS-', lpad(cast(abs(hash(seed, 'fs')) % 100 + 1 AS STRING), 4, '0'),
    '","contract_type":"',
    CASE abs(hash(seed, 'ctype')) % 3 WHEN 0 THEN 'fee_for_service' WHEN 1 THEN 'capitated' ELSE 'value_based' END,
    '","discount_pct":', cast(round((abs(hash(seed, 'disc')) % 300) / 10.0 + 10, 1) AS STRING), '}}'
  )) AS provider_data
FROM base;


## Step 5 — Operational events (400 rows)

Prior auths, appeals, grievances, and call-center events.  Each has a `event_json` STRING
column with a status history array, SLA tracking, and optional resolution object.
Good intro table — relatively flat JSON compared to claims and members.


In [ ]:
%sql
DROP TABLE IF EXISTS ${catalog_name}.ops_json_demo.operational_events;

CREATE TABLE ${catalog_name}.ops_json_demo.operational_events (
  event_id          STRING     COMMENT 'Unique event identifier (EVT-NNNNNNNN)',
  event_type        STRING     COMMENT 'prior_auth, appeal, grievance, or call_center',
  source_system     STRING     COMMENT 'Originating system name',
  event_timestamp   TIMESTAMP  COMMENT 'When the event occurred',
  related_claim_id  STRING     COMMENT 'FK to claims_submissions.claim_id (nullable)',
  related_member_id STRING     COMMENT 'FK to member_profiles.member_id',
  event_json        STRING     COMMENT 'Event payload JSON: status, priority, assigned_to, requested_service, status_history array, sla object, resolution object'
) COMMENT 'Operational events from UM, appeals, grievances, and call center with JSON status histories and SLA tracking';


In [ ]:
%sql
INSERT INTO ${catalog_name}.ops_json_demo.operational_events
WITH event_ids AS (SELECT explode(sequence(1, 400)) AS rn),
base AS (
  SELECT rn,
    concat('EVT-', lpad(cast(rn AS STRING), 8, '0')) AS event_id,
    CASE abs(hash(rn, 'type')) % 4 WHEN 0 THEN 'prior_auth' WHEN 1 THEN 'appeal' WHEN 2 THEN 'grievance' ELSE 'call_center' END AS event_type,
    CASE abs(hash(rn, 'src')) % 4 WHEN 0 THEN 'UM_Platform' WHEN 1 THEN 'CRM_System' WHEN 2 THEN 'Appeals_Tracker' ELSE 'Call_Center_IVR' END AS source_system,
    cast(concat(cast(date_add('2024-01-01', abs(hash(rn, 'dt')) % 365) AS STRING), ' ',
      lpad(cast(abs(hash(rn, 'hr')) % 12 + 7 AS STRING), 2, '0'), ':',
      lpad(cast(abs(hash(rn, 'mn')) % 60 AS STRING), 2, '0'), ':',
      lpad(cast(abs(hash(rn, 'sc')) % 60 AS STRING), 2, '0')) AS TIMESTAMP) AS event_timestamp,
    CASE WHEN abs(hash(rn, 'clm')) % 100 < 70
      THEN concat('CLM-2024-', lpad(cast(abs(hash(rn, 'clmid')) % 500 + 1 AS STRING), 5, '0'))
      ELSE NULL END AS related_claim_id,
    concat('MBR-', lpad(cast(abs(hash(rn, 'mbr')) % 300 + 1 AS STRING), 6, '0')) AS related_member_id,
    rn AS seed
  FROM event_ids
)
SELECT event_id, event_type, source_system, event_timestamp, related_claim_id, related_member_id,
  concat(
    '{"status":"', CASE abs(hash(seed, 'st')) % 4 WHEN 0 THEN 'open' WHEN 1 THEN 'in_review' WHEN 2 THEN 'resolved' ELSE 'pending' END,
    '","priority":"', CASE abs(hash(seed, 'pri')) % 3 WHEN 0 THEN 'high' WHEN 1 THEN 'medium' ELSE 'low' END,
    '","assigned_to":"',
    CASE abs(hash(seed, 'asgn')) % 6
      WHEN 0 THEN 'UM_Team_A' WHEN 1 THEN 'UM_Team_B' WHEN 2 THEN 'Appeals_Unit'
      WHEN 3 THEN 'Grievance_Unit' WHEN 4 THEN 'Call_Center_Tier1' ELSE 'Call_Center_Tier2'
    END, '"',
    CASE WHEN event_type = 'prior_auth' THEN
      concat(',"requested_service":{"procedure_code":"',
        CASE abs(hash(seed, 'proc')) % 6
          WHEN 0 THEN '27447' WHEN 1 THEN '43239' WHEN 2 THEN '93306'
          WHEN 3 THEN '70553' WHEN 4 THEN '27130' ELSE '33533'
        END,
        '","description":"',
        CASE abs(hash(seed, 'proc')) % 6
          WHEN 0 THEN 'Total knee replacement' WHEN 1 THEN 'Upper GI endoscopy'
          WHEN 2 THEN 'Echocardiogram complete' WHEN 3 THEN 'Brain MRI with contrast'
          WHEN 4 THEN 'Total hip replacement' ELSE 'CABG triple bypass'
        END,
        '","medical_necessity_code":"', CASE abs(hash(seed, 'mn')) % 3 WHEN 0 THEN 'MN-001' WHEN 1 THEN 'MN-002' ELSE 'MN-003' END, '"}')
    ELSE '' END,
    ',"status_history":[{"status":"submitted","changed_by":"system","changed_at":"', cast(event_timestamp AS STRING), '","notes":"Initial submission"}',
    CASE WHEN abs(hash(seed, 'sh1')) % 100 < 70 THEN
      concat(',{"status":"in_review","changed_by":"',
        CASE abs(hash(seed, 'rev')) % 4 WHEN 0 THEN 'reviewer_jones' WHEN 1 THEN 'reviewer_smith' WHEN 2 THEN 'reviewer_patel' ELSE 'reviewer_kim' END,
        '","changed_at":"', cast(event_timestamp + INTERVAL '2' HOUR AS STRING), '","notes":"Assigned for clinical review"}')
    ELSE '' END,
    CASE WHEN abs(hash(seed, 'sh2')) % 100 < 50 THEN
      concat(',{"status":"resolved","changed_by":"',
        CASE abs(hash(seed, 'rslv')) % 3 WHEN 0 THEN 'mgr_williams' WHEN 1 THEN 'mgr_davis' ELSE 'mgr_garcia' END,
        '","changed_at":"', cast(event_timestamp + INTERVAL '48' HOUR AS STRING), '","notes":"',
        CASE abs(hash(seed, 'note')) % 3 WHEN 0 THEN 'Approved after clinical review' WHEN 1 THEN 'Denied - does not meet criteria' ELSE 'Resolved per member request' END, '"}')
    ELSE '' END, ']',
    ',"sla":{"target_hours":', CASE event_type WHEN 'prior_auth' THEN '72' WHEN 'appeal' THEN '720' WHEN 'grievance' THEN '720' ELSE '24' END,
    ',"actual_hours":', cast(round((abs(hash(seed, 'slah')) % 12000) / 100.0 + 1, 1) AS STRING),
    ',"met":', CASE WHEN abs(hash(seed, 'slam')) % 100 < 78 THEN 'true' ELSE 'false' END, '}',
    CASE WHEN abs(hash(seed, 'res')) % 100 < 50 THEN
      concat(',"resolution":{"outcome":"',
        CASE abs(hash(seed, 'out')) % 4 WHEN 0 THEN 'approved' WHEN 1 THEN 'denied' WHEN 2 THEN 'withdrawn' ELSE 'approved_modified' END,
        '","resolved_at":"', cast(event_timestamp + INTERVAL '72' HOUR AS STRING),
        '","resolved_by":"', CASE abs(hash(seed, 'rby')) % 3 WHEN 0 THEN 'mgr_williams' WHEN 1 THEN 'mgr_davis' ELSE 'mgr_garcia' END, '"}')
    ELSE ',"resolution":null' END, '}'
  ) AS event_json
FROM base;


## Step 6 — Verify row counts

All four tables should show data.  Expected: claims=500, members=300, providers=200, ops=400.


In [ ]:
%sql
SELECT 'claims'   AS table_name, count(*) AS rows FROM ${catalog_name}.claims_json_demo.claims_submissions
UNION ALL
SELECT 'members',  count(*) FROM ${catalog_name}.member_json_demo.member_profiles
UNION ALL
SELECT 'providers', count(*) FROM ${catalog_name}.provider_json_demo.provider_network
UNION ALL
SELECT 'ops_events', count(*) FROM ${catalog_name}.ops_json_demo.operational_events
ORDER BY table_name;


## ✅  Setup complete

Open **`01_json_demo.ipynb`** to start the tutorial.

---

## Optional — Teardown / Reset

Run the cell below **only** if you want to drop all four schemas and start fresh.
It is commented out by default.


In [ ]:
%sql
-- ⚠️  OPTIONAL TEARDOWN — uncomment to drop all demo schemas and start fresh
-- DROP SCHEMA IF EXISTS ${catalog_name}.claims_json_demo  CASCADE;
-- DROP SCHEMA IF EXISTS ${catalog_name}.member_json_demo  CASCADE;
-- DROP SCHEMA IF EXISTS ${catalog_name}.provider_json_demo CASCADE;
-- DROP SCHEMA IF EXISTS ${catalog_name}.ops_json_demo     CASCADE;
